# WordPiece 分词算法（BERT 系列）

源码导航：[`core/tokenizer/wordpiece.py`](../../../core/tokenizer/wordpiece.py)。

## 1. 理论背景

**WordPiece** 是 Google 为 BERT 等模型设计的子词分割算法。与 BPE 的频次驱动不同，WordPiece 的合并策略以**语言模型似然率提升**为目标，优先合并"互信息最高"的 pair。

### 合并准则

对 pair $(a, b)$，BPE 选取 $\text{freq}(a, b)$ 最大的 pair；WordPiece 选取如下 **Score** 最大的 pair：

$$\text{Score}(a, b) = \frac{\text{freq}(a, b)}{\text{freq}(a) \times \text{freq}(b)}$$

直观理解：Score 等比于 $a$ 和 $b$ 的**点互信息（PMI）**的指数形式，度量两者共现的超预期程度。合并 Score 高的 pair 使语言模型对数似然率的提升最大。

### 词内续接前缀 `##`

WordPiece 使用 `##` 标记非词首子词（区别于 BPE 使用词尾 `</w>`）：
- `play`：词首子词（或完整词）
- `##ing`：只能跟在其他子词之后

### 编码：最长前缀匹配（MaxMatch）

WordPiece 不保存合并规则，只保存最终词表。编码时从左到右贪心查找最长词表匹配：

```
start = 0
while start < len(word):
    找最大 end 使 word[start:end] ∈ vocab  # 首部分不加 ##，后续加 ##
    若无匹配 → 整词标记为 [UNK]
```

| 算法 | BPE | WordPiece | Unigram LM |
|---|---|---|---|
| 合并准则 | $\max \text{freq}(a,b)$ | $\max \text{Score}(a,b)$ | 最优词表裁剪 |
| 编码策略 | 按 merge 顺序贪心 | MaxMatch 贪心 | Viterbi 全局最优 |
| OOV 处理 | 回退到单字符 | 整词变 `[UNK]` | 无 OOV（Unigram LM） |
| 代表模型 | GPT 系列 | BERT, DistilBERT | LLaMA, Qwen, T5 |

In [ ]:
from collections import Counter

# 模拟数据
word_freq = {"play": 10, "playing": 5, "player": 3}
tok_freq = Counter({"play": 10+5+3, "##ing": 5, "##er": 3})
pair_freq = {("play", "##ing"): 5, ("play", "##er"): 3}

def compute_score(pair, pf, tf):
    left, right = pair
    # Score = Pair 频次 / (左项频次 * 右项频次)
    return pf / (tf[left] * tf[right])

for pair, pf in pair_freq.items():
    score = compute_score(pair, pf, tok_freq)
    print(f"Pair {pair}: Score = {score:.4f}")

### 2.2 续接前缀 `##`
WordPiece 使用 `##` 来区分一个子词是否是单词的起始部分。
- `play`：独立单词或起始。
- `##ing`：只能跟在别的 Token 后面。

### 2.3 编码：最长前缀匹配 (MaxMatch)
WordPiece 不保存合并规则集（merges），而是只保存一个最终的词表（Vocab）。编码时采用从左到右的贪心匹配。

源码对应：[`_encode_word`](../../../core/tokenizer/wordpiece.py#L133)


In [ ]:
def max_match_encode(word, vocab):
    # 模拟最长前缀匹配
    ids = []
    start = 0
    while start < len(word):
        end = len(word)
        cur_token = None
        while start < end:
            sub = word[start:end]
            cand = sub if start == 0 else "##" + sub
            if cand in vocab:
                cur_token = cand
                break
            end -= 1
        
        if cur_token is None:
            return ["[UNK]"]
            
        ids.append(cur_token)
        start = end
    return ids

# 模拟一个 Vocab
vocab = {"play", "##ing", "smart", "##er", "is"}
print(f"'playing' 编码: {max_match_encode('playing', vocab)}")
print(f"'smarter' 编码: {max_match_encode('smarter', vocab)}")
print(f"'unknown' 编码: {max_match_encode('unknown', vocab)}") # 整词变 UNK

---

## 3. 训练流程模拟

WordPiece 训练时：
1. **初始化**：Vocab 里放入所有单字符（首字符不带 `##`，其它带 `##`）。
2. **循环**：
   - 计算所有相邻 pair 的 Score。
   - 选取 Score 最高的进行合并。
   - 更新 Vocab 并重写训练语料。

源码对应：`WordPieceTokenizer.train`

---

## 4. 工程实现提示

在源码 [`core/tokenizer/wordpiece.py`](../../../core/tokenizer/wordpiece.py) 中：
- **`SPECIAL_TOKENS`**：包含 BERT 常用的 `[PAD]`, `[UNK]`, `[CLS]`, `[SEP]`, `[MASK]`。
- **Lowercasing**：教学版默认会将文本转为小写，以便演示 Uncased 模型的行为。
- **贪心查找**：`_encode_word` 实现了一个双层嵌套循环来高效执行 MaxMatch。

---

## 5. 延伸阅读与参考资料

### 核心论文 (Paper)
- **Google NMT (WordPiece 详细介绍)**: Wu et al., 2016. *Google's Neural Machine Translation System*. [arXiv:1609.08144](https://arxiv.org/abs/1609.08144)
- **BERT 论文**: Devlin et al., 2018. [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)

### 优质博客 (Blog)
- **Hugging Face**: *WordPiece tokenization*. [NLP Course Chapter 6](https://huggingface.co/learn/nlp-course/chapter6/6)

### 代码库参考 (Code)
- **Google BERT 官方代码**: [google-research/bert/tokenization.py](https://github.com/google-research/bert/blob/master/tokenization.py)

---
> 父文档：[← 分词器总览](index.ipynb)